# Main Agent 실험

Main Supervisor부터 Domain Agent, 검색과 최종 답변까지 전체 흐름을 확인한다.

준비 사항:

- 저장소 루트의 `.env`에 `CLOVASTUDIO_*`, `QDRANT_*` 설정하기
- 대상 Qdrant collection에 문서 인덱스가 적재되어 있는지 확인하기
- 저장소 루트에서 `uv run --locked --group notebook jupyter lab` 실행하기

Runtime은 한 번만 만들어 재사용한다. 커밋 전에는 검색 원문과 답변 출력을 모두 지운다.

In [ ]:
# Python 경로와 .env 위치 확인용
import sys
from pathlib import Path
from pprint import pprint

# 전체 Agent Runtime과 최종 API 응답 조립기
from pension_agent.agent.runtime import build_runtime_answer_service
from pension_agent.api.presentation import build_answer_response

## 실행 환경 확인

`.venv`와 저장소 루트의 `.env`를 정상적으로 사용하는지 확인한다.

In [ ]:
# .env 내용은 출력하지 않고 파일 존재 여부만 본다.
project_root = Path.cwd()
{
    "python": sys.executable,
    "cwd": str(project_root),
    "env_file_exists": (project_root / ".env").is_file(),
}

## 전체 Agent Runtime 조립

HCX, Domain Agent, Search Service와 Qdrant를 한 번에 준비한다.

In [ ]:
# 셀을 다시 실행하면 기존 Runtime을 먼저 닫는다.
existing_answer_service = globals().get("answer_service")
if existing_answer_service is not None:
    await existing_answer_service.aclose()

# HCX, embedding, Qdrant와 Agent 그래프를 조립한다.
answer_service = await build_runtime_answer_service()
"Agent Runtime 준비 완료"

## 질문 실행 함수

질문을 실행하고 최종 API 응답과 내부 DomainResult를 함께 돌려준다.

In [ ]:
async def ask_main_agent(*, question_id: str, question: str) -> dict[str, object]:
    # Main Supervisor에 질문을 전달한다.
    result = await answer_service.run(
        question_id=question_id,
        question=question,
    )
    # 평가 API와 같은 5개 필드로 변환한다.
    response = build_answer_response(result)
    return {
        "response": response.model_dump(mode="json"),
        "domain_results": result.state["domain_results"],
    }

## 질문 하나 실행

먼저 최종 API 응답을 확인한다.

In [ ]:
# question_id는 실험 결과를 구분할 수 있게 정한다.
run = await ask_main_agent(
    question_id="NB-001",
    question="IRP와 연금저축의 차이와 선택할 때 확인할 조건을 알려줘",
)
# 최종 5개 API 필드를 본다.
pprint(run["response"])

In [ ]:
# 호출된 도메인과 선택된 근거를 본다.
pprint(run["domain_results"])

## 여러 질문 순차 실행

필요한 질문만 남기고 API 제한을 고려해 하나씩 실행한다.

In [ ]:
# 실험할 질문만 목록에 남긴다.
questions = [
    "IRP 계좌를 다른 금융회사로 이전하려면 어떤 조건을 확인해야 해?",
    "연금 수령 시 세금 판단에 필요한 조건은 무엇이야?",
    "연금 상품의 비용과 위험을 비교할 때 무엇을 확인해야 해?",
]
batch_responses = []
# rate limit을 피하려고 순차로 호출한다.
for index, question in enumerate(questions, start=1):
    batch_run = await ask_main_agent(
        question_id=f"NB-BATCH-{index:03d}",
        question=question,
    )
    batch_responses.append(batch_run["response"])

batch_responses

## Runtime 정리

실험을 마치면 외부 연결을 닫는다. 다시 질문하려면 Runtime 조립부터 실행한다.

In [ ]:
# HCX, embedding과 Qdrant client를 정리한다.
await answer_service.aclose()
answer_service = None
"Agent Runtime 종료 완료"